# 01 — Clean raw data

Turns the three raw downloads into the clean tables every later step reads.

| | |
|---|---|
| **Reads** | `raw/branches/Libraries_*.csv`, `raw/boundaries/Boundaries - City.geojson`, `raw/boundaries/tl_2021_17_tract.zip`, `raw/demographics/R13395758_SL140.csv` |
| **Writes** | `clean/branches.csv`, `clean/population_demographics.csv`, `clean/census_demos.csv` |
| **Needs** | `cpl-s3` |

This is the same work the `cpl-cleaning` KFP pipeline does, with the
`@dsl.component` wrappers stripped off so you can run it cell by cell.

### Running this as an Elyra node

Drag this notebook onto a pipeline canvas, then open its **node properties**:

| Property | Value |
|---|---|
| Runtime image | Datascience with Python 3.12 (or any image with pip access) |
| Kubernetes secret | `cpl-s3` → `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET` |

The notebook installs its own dependencies in the first cell, so any runtime
image works. Everything it reads and writes lives in S3 — nothing is passed
between nodes as a file, so node ordering is the only dependency.

In [ ]:
%pip install -q boto3 geopandas pyogrio pandas

In [ ]:
import io
import os
import logging

import boto3
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)
log = logging.getLogger("clean_data")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
BUCKET = os.environ["AWS_S3_BUCKET"]
log.info("Bucket: %s", BUCKET)

## Branches

The raw export has a few misspelled branch names and stores coordinates as a
`"(lat, lon)"` string. Roosevelt closed but still shows up in historical
coverage, so it is added back with `PERMANENTLY_CLOSED = 1` rather than dropped.

In [ ]:
import ast

raw_key = "raw/branches/Libraries_-_Locations___Contact_Information__and_Usual_Hours_of_Operation.csv"
obj = s3.get_object(Bucket=BUCKET, Key=raw_key)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

df.columns = df.columns.str.strip()
df = df.rename(columns={"NAME": "BRANCH"})

name_fixes = {
    "Daley, Richard J.": "Daley, Richard J. - Bridgeport",
    "Daley, Richard M.": "Daley, Richard M. - W. Humboldt",
    "Harold Washtington Library Center": "Harold Washington Library Center",
    "King": "Martin Luther King Jr.",
}
df["BRANCH"] = df["BRANCH"].replace(name_fixes)
df["PERMANENTLY_CLOSED"] = 0

roosevelt = pd.DataFrame([{
    "BRANCH": "Roosevelt",
    "HOURS OF OPERATION": None,
    "ADDRESS": "1101 W. Taylor Street",
    "CITY": "Chicago", "STATE": "IL", "ZIP": "60607",
    "PHONE": None, "WEBSITE": None,
    "LOCATION": "(41.86943838326072, -87.6544586134924)",
    "PERMANENTLY_CLOSED": 1,
}])
df = pd.concat([df, roosevelt], ignore_index=True)

coords = df["LOCATION"].apply(ast.literal_eval)
df["LATITUDE"] = coords.apply(lambda c: float(c[0]))
df["LONGITUDE"] = coords.apply(lambda c: float(c[1]))

branches = df.sort_values("BRANCH").reset_index(drop=True)

buf = io.BytesIO()
branches.to_csv(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="clean/branches.csv", Body=buf.getvalue())
log.info("Wrote %d branches to s3://%s/clean/branches.csv", len(branches), BUCKET)

branches[["BRANCH", "LATITUDE", "LONGITUDE", "PERMANENTLY_CLOSED"]].head()

## Census tracts

Illinois ships tracts for the whole state, so they get clipped to the Chicago
city boundary, then joined to the demographic export on FIPS. Output keeps the
population columns plus the tract polygon as WKT.

In [ ]:
import tempfile

import geopandas as gpd

obj = s3.get_object(Bucket=BUCKET, Key="raw/boundaries/Boundaries - City.geojson")
chicago = gpd.read_file(io.BytesIO(obj["Body"].read()))

obj = s3.get_object(Bucket=BUCKET, Key="raw/boundaries/tl_2021_17_tract.zip")
with tempfile.NamedTemporaryFile(suffix=".zip") as tmp:
    tmp.write(obj["Body"].read())
    tmp.flush()
    il_tracts = gpd.read_file(tmp.name)

il_tracts = il_tracts.to_crs(chicago.crs)
chicago_tracts = gpd.clip(il_tracts, chicago)
log.info("Clipped %d Illinois tracts down to %d in Chicago", len(il_tracts), len(chicago_tracts))

obj = s3.get_object(Bucket=BUCKET, Key="raw/demographics/R13395758_SL140.csv")
demos_raw = pd.read_csv(io.BytesIO(obj["Body"].read()))
demos_raw = demos_raw.iloc[1:, :]  # second header row

merged = chicago_tracts.merge(demos_raw, how="left", left_on="GEOID", right_on="FIPS")

id_cols = {
    "GEOID": "geoid",
    "Qualifying Name": "qualifying name",
    "ALAND": "current land area (sq. m)",
    "AWATER": "current water area (sq. m)",
}
merged = merged.rename(columns=id_cols)

pop_cols = [c for c in merged.columns if c.startswith("Total Population") or
            c.startswith("Population Density") or c.startswith("Total:")]

keep = list(id_cols.values()) + pop_cols + ["geometry"]
keep = [c for c in keep if c in merged.columns]
tracts = merged[keep].copy()
tracts.columns = tracts.columns.str.lower()

buf = io.BytesIO()
tracts.to_csv(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="clean/population_demographics.csv", Body=buf.getvalue())
log.info("Wrote %d tracts to s3://%s/clean/population_demographics.csv", len(tracts), BUCKET)

tracts.head()

## Tract demographics

Same join, but keeping the percentage columns (poverty, internet access,
commute) instead of the raw population counts. The need-score work reads this one.

In [ ]:
merged2 = chicago_tracts.merge(demos_raw, how="left", left_on="GEOID", right_on="FIPS")

keep_cols = ["GEOID", "Qualifying Name", "geometry", "Total Population"]
pct_cols = [c for c in merged2.columns if c.startswith("Percent") or c.startswith("average commute")]
keep_cols.extend(pct_cols)
keep_cols = [c for c in keep_cols if c in merged2.columns]
census_demos = merged2[keep_cols].copy()

census_demos = census_demos.rename(columns={
    "GEOID": "geoid",
    "Qualifying Name": "qualifying name",
    "Total Population": "total population",
})

for col in census_demos.columns:
    if col.startswith("Percent") or col.startswith("average"):
        census_demos[col] = pd.to_numeric(census_demos[col], errors="coerce")

buf = io.BytesIO()
census_demos.to_csv(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="clean/census_demos.csv", Body=buf.getvalue())
log.info("Wrote %d tract demographics to s3://%s/clean/census_demos.csv", len(census_demos), BUCKET)

census_demos.head()